Set $\mu_A$, $M$, and the training parameters.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from utils_nre import Classifier, NRE, sample, predict, train_nre
from utils_preselection import train_selection, calibrate_cut, selected_yields
from utils_nre_inference import (
    finite_reference_asimov, simulator_asimov, compress_reference,
    simulator_probabilities, run_toys,
)
from utils_nre_plotting import plot_training, plot_asimov_scans, plot_q0

DEVICE = "cuda"  # Use "cpu" for CPU execution.
TRAIN = True  # Set False to load the saved selection and NRE weights.
WORK_DIR = Path("workspace")
WORK_DIR.mkdir(exist_ok=True)
SEED = 120_926
MU_A = 1.0

SELECTION_CONFIG = dict(
    widths=[256] * 3, epochs=30, batch_size=2048, learning_rate=1e-3,
    scheduler_step=30, scheduler_gamma=1.0, patience=8,
)
N_SELECTION_TRAIN = 500_000
N_SELECTION_VALIDATION = 100_000
N_CALIBRATION = 5_000_000
N_YIELD = 5_000_000
INCLUSIVE_YIELDS = np.array([1100.0, 1_000_000.0])
TARGET_B_TO_S = 250.0

NRE_CONFIG = dict(
    widths=[1024] * 4, epochs=140, batch_size=4096, learning_rate=1e-4,
    scheduler_step=20, scheduler_gamma=0.1,
)
ENSEMBLE_SIZE = 4
N_TRAIN = 5_000_000
N_VALIDATION = 500_000
MC_SIZES = np.array([256, 1024, 4096, 16384, 65536, 262144, 1048576, 2000000])
N_REPETITIONS = 8
N_REFERENCE = 5_000_000
N_SIMULATOR = 5_000_000
N_TOYS = 100_000
N_BINS = 1024
SCAN_MU = np.linspace(0, 2, 161)

Fix the selection and estimate $\lambda_S$ and $\lambda_B$.

In [ ]:
if TRAIN:
    classifier, history = train_selection(
        N_SELECTION_TRAIN, N_SELECTION_VALIDATION, SELECTION_CONFIG, DEVICE, SEED + 900_000,
    )
    log_cut = calibrate_cut(
        classifier, N_CALIBRATION, INCLUSIVE_YIELDS, TARGET_B_TO_S, SEED + 900_001,
    )
    selection = lambda x: predict(classifier, x) >= log_cut
    lambda_s = selected_yields(selection, N_YIELD, INCLUSIVE_YIELDS, SEED + 900_002)
    torch.save(dict(weights=classifier.state_dict(), log_cut=log_cut,
                    lambda_s=lambda_s.tolist()), WORK_DIR / "selection.pt")
    pd.DataFrame(history).to_csv(WORK_DIR / "selection_training.csv", index=False)
else:
    saved = torch.load(WORK_DIR / "selection.pt", map_location=DEVICE, weights_only=True)
    classifier = Classifier(SELECTION_CONFIG["widths"]).to(DEVICE)
    classifier.load_state_dict(saved["weights"])
    classifier.eval()
    log_cut, lambda_s = saved["log_cut"], np.array(saved["lambda_s"])
    selection = lambda x: predict(classifier, x) >= log_cut
print("lambda_S, lambda_B:", lambda_s)

Train $r_{s,\boldsymbol\psi}$ using $p_{\rm ref}=(p_S+p_B)/2$.

In [ ]:
if TRAIN:
    training = {
        s: sample(s, N_TRAIN, selection, np.random.default_rng(SEED + i))
        for i, s in enumerate(("S", "B", "ref"))
    }
    validation = {
        s: sample(s, N_VALIDATION, selection, np.random.default_rng(SEED + 100 + i))
        for i, s in enumerate(("S", "B", "ref"))
    }
    nre, history = train_nre(training, validation, NRE_CONFIG, ENSEMBLE_SIZE, DEVICE, SEED + 200)
    torch.save(nre.state_dict(), WORK_DIR / "nre.pt")
    history = pd.DataFrame(history)
    history.to_csv(WORK_DIR / "nre_training.csv", index=False)
    del training, validation
else:
    nre = NRE(NRE_CONFIG["widths"], ENSEMBLE_SIZE).to(DEVICE)
    nre.load_state_dict(torch.load(WORK_DIR / "nre.pt", map_location=DEVICE, weights_only=True))
    nre.eval()
    history = pd.read_csv(WORK_DIR / "nre_training.csv")
plot_training(history)
plt.show()

Fix $Z_s$ on a large $\mathcal X_M$ for conventional samples and toy fits.

In [ ]:
def draw_ratios(component, n, seed):
    x = sample(component, n, selection, np.random.default_rng(seed))
    return predict(nre, x)

r_reference = draw_ratios("ref", N_REFERENCE, SEED + 1000)
Z = r_reference.mean(axis=0)
reference_asimov = finite_reference_asimov(r_reference, lambda_s, MU_A, SCAN_MU)

Compare conventional samples with $w_m^A=\omega_m h(\mathbf{x}_m;\mu_A)$, recomputing $Z_s$ on each $\mathcal X_M$.

In [ ]:
study = {"simulator": [], "normalized": []}
for repetition in range(N_REPETITIONS):
    seed = SEED + 10_000 + 10 * repetition
    r_S = draw_ratios("S", MC_SIZES[-1] // 2, seed)
    r_B = draw_ratios("B", MC_SIZES[-1] // 2, seed + 1)
    r_ref = draw_ratios("ref", MC_SIZES[-1], seed + 2)
    study["simulator"].append([
        simulator_asimov(r_S[:M // 2], r_B[:M // 2], lambda_s, Z, MU_A, SCAN_MU)
        for M in MC_SIZES
    ])
    study["normalized"].append([
        finite_reference_asimov(r_ref[:M], lambda_s, MU_A, SCAN_MU)
        for M in MC_SIZES
    ])
    print(f"Asimov repetition {repetition + 1}/{N_REPETITIONS}")

table = pd.DataFrame([
    dict(construction=kind, repetition=i, M=M, mu_hat=result["mu_hat"], q0_A=result["q0_A"])
    for kind, repetitions in study.items()
    for i, results in enumerate(repetitions)
    for M, result in zip(MC_SIZES, results)
])
table.to_csv(WORK_DIR / "asimov.csv", index=False)
table.groupby(["construction", "M"])[["mu_hat", "q0_A"]].agg(["mean", "std"])

Plot $t_{\mu,A}$ for each $M$.

In [ ]:
plot_asimov_scans(study["simulator"][0], MC_SIZES, SCAN_MU, "Simulator samples")
plot_asimov_scans(study["normalized"][0], MC_SIZES, SCAN_MU, "Normalized Asimov samples")
plt.show()

Generate NRE-model and simulator-bank pseudo-experiments; fit $\widehat\mu$ and $q_0$ using a binned approximation to $h$.

In [ ]:
r_S_toys = draw_ratios("S", N_SIMULATOR, SEED + 50_000)
r_B_toys = draw_ratios("B", N_SIMULATOR, SEED + 50_001)
edges, model_probabilities, bin_ratio = compress_reference(r_reference, lambda_s, Z, N_BINS)
sim_probabilities = simulator_probabilities([r_S_toys, r_B_toys], lambda_s, Z, edges)
model_toys = run_toys(bin_ratio, model_probabilities, lambda_s, MU_A, N_TOYS, SEED + 70_000)
simulator_toys = run_toys(bin_ratio, sim_probabilities, lambda_s, MU_A, N_TOYS, SEED + 70_001)
pd.DataFrame(model_toys).to_csv(WORK_DIR / "model_toys.csv", index=False)
pd.DataFrame(simulator_toys).to_csv(WORK_DIR / "simulator_toys.csv", index=False)

Compare $q_0$ with the Wald predictions from $q_{0,A}$.

In [ ]:
shown = [0, len(MC_SIZES) // 2, len(MC_SIZES) - 1]
labels = [f"M = {MC_SIZES[i]:,}" for i in shown]
plot_q0(model_toys, simulator_toys,
        [study["simulator"][0][i] for i in shown], labels, "Simulator predictions")
plot_q0(model_toys, simulator_toys,
        [study["normalized"][0][i] for i in shown] + [reference_asimov],
        labels + [f"M = {N_REFERENCE:,}"], "Normalized Asimov predictions")
plt.show()